In [72]:
import sys
sys.path.append('/home/cyf/wbi/Virginia/code')
import torch
from CoarseFlow.models.SparseGMFlow3D import CoarseMatchingNetV3
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 
from CoarseFlow.training.train import train_coarse_matching_model,inference
from CoarseFlow.datasets.synthetic_dataset import SparseZStackSyntheticDataset, CachedDataset
from wbi_0123.wholistic_registration.src.wholistic_registration.utils.simulation import generateMotion_Biophysical
from wbi_0123.wholistic_registration.src.wholistic_registration.utils import calFlow3d_Wei_v1, calFlowCrossResolution
from wbi_0123.wholistic_registration.src.wholistic_registration.utils import IO
gut_Path     = "/home/cyf/wbi/Virginia/registrated_data/f260201/gut/raw/260201_test1_0_00002_TZCXY.ome.tif"
ventral_Path = "/home/cyf/wbi/Virginia/registrated_data/f260201/ventral/raw/260201_test1_0_00003_TZCXY.ome.tif"
dorsal_Path  = "/home/cyf/wbi/Virginia/registrated_data/f260201/dorsal/raw/260201_test1_0_00005_TZCXY.ome.tif"

gut_ref,gut_ref_desc = IO.readTiff(gut_Path)
ventral_ref,ventral_ref_desc = IO.readTiff(ventral_Path)
dorsal_ref,dorsal_ref_desc = IO.readTiff(dorsal_Path)

gut_volume = gut_ref.transpose(0,2,3,1)[2,:,:,70:90]
ventral_volume = ventral_ref.transpose(0,2,3,1)[2,:,:,50:75]
dorsal_volume = dorsal_ref.transpose(0,2,3,1)[2,:,:,40:110]

F2013_path = '/home/cyf/wbi/Virginia/raw_data/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15846.nd2'
F2013 = IO.readND2Frame(F2013_path, 0, channel=1).squeeze().transpose(2,1,0)

volumes= [F2013]

base_dataset = SparseZStackSyntheticDataset(
    volumes=volumes,
    motion_fn=generateMotion_Biophysical,
    warp_fn=calFlow3d_Wei_v1.correctMotion,
    num_sparse_slices=3,
    control_stride=16,
    ref_spacing=(1.0, 1.0, 1.0),
    mov_spacing=(1.0, 1.0, 5.0),
    num_samples_per_volume=100,
    motion_kwargs=dict(
        art_R_xy=120,
        amp_xy=8,
        zRatio=3,
        use_incompressibility=True,
    ),
    noise_std=0.00,
    normalize=True,
    gt_direction="mov_to_ref",
)



Helper function
---

Inference & Evaluation
---

In [73]:
################  with valid mask
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = "/home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarse_matching_v4_gut/latest.pth"
model_valid_mask = CoarseMatchingNetV3(
    dim=64,
    radius=(2, 4, 4),          # 必须和训练时一致
    temperature=0.05,          # 必须和训练时一致
    use_learned_matching=False,
    control_stride=16,
    num_refine_iters=1,
    encoder_stride=4,
    query_chunk_size=1024,
).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model_valid_mask.load_state_dict(ckpt["model"])
model_valid_mask.eval()
print("Loaded checkpoint from epoch:", ckpt.get("epoch", "unknown"))


Loaded checkpoint from epoch: 390


In [80]:
import numpy as np
sample = base_dataset[0]
result = inference(
    model = model_valid_mask,
    sample_or_batch=sample,
    device = device
)

_, K, Y, X = sample["mov"].shape
print(result['metrics'])
pred_phase = result['pred_coords'].detach().cpu().numpy().squeeze()
z_init = sample["z_init"].detach().cpu().numpy()
gt_phase= sample['gt_coords'].detach().cpu().numpy().squeeze()
dat_ref = volumes[0]
H_ref_layer = calFlowCrossResolution.generate_continuous_H_gpu(dat_ref, zRatio=3)
pred_phase_dense_xyz = control_coords_to_dense_coords(
    pred_coords=pred_phase,
    z_init=z_init,
    image_shape_yx=(Y, X),
    control_stride=16,
    input_order="zyx",
    output_order="xyz",
).transpose(2,1,0,3)
gt_phase_dense_xyz = control_coords_to_dense_coords(
    pred_coords=gt_phase,
    z_init=z_init,
    image_shape_yx=(Y, X),
    control_stride=16,
    input_order="zyx",
    output_order="xyz",
).transpose(2,1,0,3)
dat_cor = calFlowCrossResolution.apply_H_to_matrix_gpu(pred_phase_dense_xyz, H_ref_layer).get()
dat_mov = calFlowCrossResolution.apply_H_to_matrix_gpu(gt_phase_dense_xyz, H_ref_layer).get()
print(f"RMSE:{np.mean(np.abs(dat_cor-dat_mov))}")
print(f"raw RMSE:{np.mean(np.abs(dat_ref[:,:,[int(z_init[i]) for i in range(len(z_init))]]-dat_mov))}")
# print(f"SSIM:{}")
# from wbi_0123.wholistic_registration.src.wholistic_registration.utils import visualization
# visualization.visualize_2d_image(dat_mov[:,:,0],title = "dat_mov")
# visualization.visualize_2d_image(dat_cor[:,:,0],title = "dat_cor")
# visualization.visualize_2d_image(dat_ref[:,:,int(z_init[0])],title = "dat_ref")

{'disp_mae_all': tensor(2.1996), 'disp_mae_valid': tensor(2.1059), 'z_disp_mae_all': tensor(1.1284), 'z_disp_mae_valid': tensor(1.0024), 'coord_mae_all': tensor(2.1996), 'coord_mae_valid': tensor(2.1059), 'z_coord_mae_all': tensor(1.1284), 'z_coord_mae_valid': tensor(1.0024), 'valid_ratio': tensor(0.5439), 'prob_max': tensor(0.0545), 'entropy': tensor(4.3469)}
RMSE:31.204288482666016
raw RMSE:37.488285064697266


In [75]:
################  no mask
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = "/home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarse_matching_v3_gut/latest.pth"
model_no_mask = CoarseMatchingNetV3(
    dim=64,
    radius=(2, 4, 4),          # 必须和训练时一致
    temperature=0.05,          # 必须和训练时一致
    use_learned_matching=False,
    control_stride=16,
    num_refine_iters=1,
    encoder_stride=4,
    query_chunk_size=1024,
).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model_no_mask.load_state_dict(ckpt["model"])
model_no_mask.eval()
print("Loaded checkpoint from epoch:", ckpt.get("epoch", "unknown"))


Loaded checkpoint from epoch: 401


In [81]:
import numpy as np
sample = base_dataset[0]
result = inference(
    model = model_no_mask,
    sample_or_batch=sample,
    device = device
)

_, K, Y, X = sample["mov"].shape
print(result['metrics'])
pred_phase = result['pred_coords'].detach().cpu().numpy().squeeze()
z_init = sample["z_init"].detach().cpu().numpy()
gt_phase= sample['gt_coords'].detach().cpu().numpy().squeeze()
dat_ref = F2013
H_ref_layer = calFlowCrossResolution.generate_continuous_H_gpu(dat_ref, zRatio=3)
pred_phase_dense_xyz = control_coords_to_dense_coords(
    pred_coords=pred_phase,
    z_init=z_init,
    image_shape_yx=(Y, X),
    control_stride=16,
    input_order="zyx",
    output_order="xyz",
).transpose(2,1,0,3)
gt_phase_dense_xyz = control_coords_to_dense_coords(
    pred_coords=gt_phase,
    z_init=z_init,
    image_shape_yx=(Y, X),
    control_stride=16,
    input_order="zyx",
    output_order="xyz",
).transpose(2,1,0,3)
dat_cor = calFlowCrossResolution.apply_H_to_matrix_gpu(pred_phase_dense_xyz, H_ref_layer).get()
dat_mov = calFlowCrossResolution.apply_H_to_matrix_gpu(gt_phase_dense_xyz, H_ref_layer).get()
print(f"RMSE:{np.mean(np.abs(dat_cor-dat_mov))}")
print(f"raw RMSE:{np.mean(np.abs(dat_ref[:,:,[int(z_init[i]) for i in range(len(z_init))]]-dat_mov))}")


{'disp_mae_all': tensor(2.1604), 'disp_mae_valid': tensor(2.1078), 'z_disp_mae_all': tensor(1.0945), 'z_disp_mae_valid': tensor(1.2340), 'coord_mae_all': tensor(2.1604), 'coord_mae_valid': tensor(2.1078), 'z_coord_mae_all': tensor(1.0945), 'z_coord_mae_valid': tensor(1.2340), 'valid_ratio': tensor(0.5534), 'prob_max': tensor(0.0585), 'entropy': tensor(4.2219)}
RMSE:31.56789779663086
raw RMSE:37.771873474121094


In [43]:
import numpy as np
from importlib import reload
from CoarseFlow.datasets import synthetic_dataset
reload(synthetic_dataset)
# 1. 取一个样本
sample = base_dataset[0]
dat_mov = sample["mov"].detach().cpu().numpy().squeeze().transpose(2, 1, 0)
dat_ref = sample["ref"].detach().cpu().numpy().squeeze().transpose(2, 1, 0)
print("dat_mov:", dat_mov.shape)
print("dat_ref:", dat_ref.shape)
gt_coords_ctrl_zyx = sample["gt_coords"].detach().cpu().numpy()
z_init = sample["z_init"].detach().cpu().numpy()
_, K, Y, X = sample["mov"].shape
gt_phase_dense_xyz = control_coords_to_dense_coords(
    pred_coords=gt_coords_ctrl_zyx,
    z_init=z_init,
    image_shape_yx=(Y, X),
    control_stride=16,
    input_order="zyx",
    output_order="xyz",
).transpose(2,1,0,3)
# print("gt_phase_dense_xyz before transpose:", gt_phase_dense_xyz.shape)
# motion = sample["motion"]       # (X,Y,Z,3), xyz
# z_init = sample["z_init"].numpy().astype(int)
# X, Y, Z, _ = motion.shape
# K = len(z_init)
# xx, yy = np.meshgrid(
#     np.arange(X, dtype=np.float32),
#     np.arange(Y, dtype=np.float32),
#     indexing="ij",
# )
# gt_phase_dense_xyz = np.zeros((X, Y, K, 3), dtype=np.float32)
# for kk, z in enumerate(z_init):
#     gt_phase_dense_xyz[:, :, kk, 0] = xx + motion[:, :, z, 0]
#     gt_phase_dense_xyz[:, :, kk, 1] = yy + motion[:, :, z, 1]
#     gt_phase_dense_xyz[:, :, kk, 2] = float(z) + motion[:, :, z, 2]
# print("gt_phase_dense_xyz after transpose:", gt_phase_dense_xyz.shape)
# # (X,Y,K,3)

# # 6. 用 GT phase 做校正
H_ref_layer = calFlowCrossResolution.generate_continuous_H_gpu(
    dat_ref,
    zRatio=3,
)

dat_cor_gt = calFlowCrossResolution.apply_H_to_matrix_gpu(
    pred_phase_dense_xyz,
    H_ref_layer,
)

print("dat_cor_gt:", dat_cor_gt.shape)

dat_mov: (1152, 1152, 3)
dat_ref: (1152, 1152, 19)
dat_cor_gt: (1152, 1152, 3)


In [53]:
mse_before = np.mean((dat_mov - dat_ref[:, :, z_init.astype(int)]) ** 2)
mse_after_gt = np.mean((dat_cor_gt.get() - dat_mov) ** 2)

print("MSE before:", mse_before)
print("MSE after GT phase:", mse_after_gt)

MSE before: 0.016865866
MSE after GT phase: 0.01626983
